In [1]:
import sys, os

# Repo must come *first* so we do not import an older ``gatetracker`` from site-packages.
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, "gatetracker")):
    _REPO_ROOT = _cwd
else:
    _REPO_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
if _REPO_ROOT in sys.path:
    sys.path.remove(_REPO_ROOT)
sys.path.insert(0, _REPO_ROOT)

import torch
import numpy as np
import rerun as rr
import matplotlib.pyplot as plt

from gatetracker.geometry.pipeline import GeometryPipeline
from gatetracker.data.pseudo_gt import (
    PseudoGTGenerator,
    TrajectoryConfig,
    DeformationConfig,
    GridConfig,
)

%load_ext autoreload
%autoreload 2

In [9]:
# ========================  PARAMETERS  ========================

# --- dataset (edit these) ---
DATASET_NAME = "SCARED"                                      # "SCARED" or "CHOLEC80"
DATASET_PATH = "/home/shared/nearmrs/arota/SCARED"           # <-- your path
VIDEO_NAME   = "train_v16"                                    # video folder name

IMAGE_H = 448
IMAGE_W = 448
DEVICE  = "cuda"
SEED    = 42

# --- trajectory ---
traj_cfg = TrajectoryConfig(
    n_frames=48,
    z_bias_range=(0.00, 0.00),             # +closer / -farther from surface
    complexity_range=(0.8,1),           # 0=gentle, 1=winding
    translation_range=(0.0, 0.2),        # per-frame translation amplitude
    rotation_range_deg=(0.0, 0.1),         # per-frame rotation amplitude
    forward_bias_range=(0.00, 0.01),       # endoscopic forward drift
    speed_scale_range=(0,0.5),          # global speed multiplier
    still_fraction_range=(0,0.5),      # fraction of near-still frames
)

# --- deformation ---
deform_cfg = DeformationConfig(
    n_deformers_range=(5,10),
    sigma_frac_range=(0.08, 0.5),          # kernel radius (frac of scene extent)
    amplitude_frac_range=(0.00, 0.1),     # max displacement (frac of scene extent)
    drag_weight=0.5,                         # probability weights per type
    inflate_weight=0.0,
    twist_weight=0.5,
    twist_max_deg_range=(1.0, 5.0),
    temporal_smooth_range=(2.0, 6.0),
)

# --- query grid ---
grid_cfg = GridConfig(grid_size=16, margin_frac=0.03)
from dataset import CHOLEC80, SCARED

DatasetClass = CHOLEC80 if DATASET_NAME == "CHOLEC80" else SCARED
ds = DatasetClass(
    path=DATASET_PATH,
    vids=[VIDEO_NAME],
    frameskip=[1],
    fps=2,
    height=IMAGE_H,
    width=IMAGE_W,
    with_depth=False,
    random_pose=False,
    crop_zoom_factor=1.0,
)
# print(f"Dataset length: {len(ds)}")

sample = next(iter(torch.utils.data.DataLoader(ds, batch_size=1, shuffle=True)))
I0 = sample["framestack"][:, 0].to(DEVICE)  # [1, 3, H, W]
# print(f"Source image: {I0.shape}, range [{I0.min():.3f}, {I0.max():.3f}]")
geom = GeometryPipeline(
    geometry_model_name="Ruicheng/moge-2-vits-normal",
    height=IMAGE_H,
    width=IMAGE_W,
    return_normalized_depth=False,
    device=DEVICE,
).eval()

with torch.no_grad():
    D0, N0, K = geom.compute_geometry(I0)

# print(f"Depth  : {D0.shape}, range [{D0.min():.4f}, {D0.max():.4f}]")
# print(f"K      : {K[0]}")
gen = PseudoGTGenerator(IMAGE_H, IMAGE_W, device=DEVICE)

result = gen.generate(
    image=I0,
    depth=D0,
    intrinsics=K,
    trajectory=traj_cfg,
    deformation=deform_cfg,
    grid=grid_cfg,
    visibility_z_tol_frac=0.2,
    visibility_z_abs_min=5e-3,
    visibility_depth_dilate=7,
    visibility_query_patch_rad=2,
    visibility_temporal_window=5,

)

T = result.frames.shape[0]
Q = result.tracks.shape[1]
# print(f"frames     : {result.frames.shape}")
# print(f"tracks     : {result.tracks.shape}")
# print(f"visibility : {result.visibility.shape}, avg visible: {result.visibility.float().mean():.3f}")
# print(f"poses      : {result.poses.shape}")


# # PseudoGTGenerator.log_to_rerun(result, subsample_cloud=8, n_tracks_3d=128)
# print("Logged to Rerun  --  open with:  rerun track_generation.rrd")
VIDEO_PATH = "track_generation_video.mp4"
PseudoGTGenerator.render_video(
    result, VIDEO_PATH,
    fps=10, trail_length=8, log_to_rerun=True,
    # zbuf_overlay_alpha=0.35,
)

from IPython.display import Video, display
display(Video(VIDEO_PATH, embed=True, html_attributes="controls loop autoplay muted"))

In [36]:
import rerun.blueprint as rrb

blueprint = rrb.Blueprint(
    rrb.Horizontal(
        # left: 3D world view (point cloud, camera frustum, track polylines)
        rrb.Spatial3DView(
            name="3D Scene",
            origin="world",
            contents=[
                "+ $origin/cloud",
                "+ $origin/cam_path",
                "+ $origin/tracks3d/**",
                "+ $origin/camera",
            ],
            background=[30, 30, 30],
        ),
        # right column: camera image views stacked vertically
        rrb.Vertical(
            # rendered frame with 2D track overlay
            rrb.Spatial2DView(
                name="Camera + Tracks",
                origin="world/camera",
                contents=[
                    "+ $origin/image",
                    "+ $origin/tracks2d",
                ],
            ),
            # annotated video (drawn trails) — populated by render_video()
            rrb.Spatial2DView(
                name="Annotated Video",
                origin="video/annotated",
            ),
            row_shares=[1, 1],
        ),
        column_shares=[3, 2],
    ),
    rrb.TimePanel(state="expanded"),
    rrb.BlueprintPanel(state="collapsed"),
    rrb.SelectionPanel(state="collapsed"),
)

rr.disconnect()
rr.init("tracks")
rr.send_blueprint(blueprint)
rr.serve_grpc(grpc_port=9876)


'rerun+http://127.0.0.1:9876/proxy'